# Gradient Descent Basics

This notebook demonstrates fundamental concepts of gradient descent optimization.

## Contents
1. Univariate Gradient Descent
2. Multivariate Gradient Descent
3. Learning Rate Effects
4. Visualization of Optimization Trajectories

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from gradient_descent import BatchGradientDescent, quadratic_function, quadratic_gradient
from gradient_descent import rosenbrock_function, rosenbrock_gradient

print("Libraries imported successfully!")

## 1. Univariate Gradient Descent

Let's start with a simple 1D optimization problem:

$$f(x) = x^2$$

The gradient is:

$$f'(x) = 2x$$

Gradient descent update rule:

$$x \leftarrow x - \eta f'(x) = x - 2\eta x$$

In [ ]:
# Initialize optimizer
optimizer = BatchGradientDescent(learning_rate=0.1, max_iterations=50)

# Starting point
x0 = np.array([5.0])

# Optimize
x_opt = optimizer.optimize(quadratic_function, quadratic_gradient, x0)

print(f"Starting point: {x0[0]:.4f}")
print(f"Optimized point: {x_opt[0]:.4f}")
print(f"Number of iterations: {len(optimizer.history['params'])}")

# Visualize
x_vals = np.linspace(-6, 6, 200)
y_vals = x_vals ** 2

plt.figure(figsize=(12, 6))
plt.plot(x_vals, y_vals, 'b-', linewidth=2, label='$f(x) = x^2$')

# Plot optimization path
params = [p[0] for p in optimizer.history['params']]
losses = optimizer.history['loss']
plt.plot(params, losses, 'ro-', markersize=6, label='Optimization Path')
plt.plot(params[0], losses[0], 'go', markersize=12, label='Start')
plt.plot(params[-1], losses[-1], 'r*', markersize=15, label='End')

plt.xlabel('x', fontsize=12)
plt.ylabel('f(x)', fontsize=12)
plt.title('1D Gradient Descent on Quadratic Function', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 2. Multivariate Gradient Descent

Now let's tackle the 2D Rosenbrock function (banana function):

$$f(x, y) = (1-x)^2 + 100(y-x^2)^2$$

This is a challenging optimization problem with a narrow valley.

Global minimum at $(1, 1)$ with $f(1,1) = 0$

In [ ]:
# Initialize optimizer
optimizer = BatchGradientDescent(learning_rate=0.001, max_iterations=2000, tolerance=1e-8)

# Starting point
x0 = np.array([-1.0, -1.0])

# Optimize
x_opt = optimizer.optimize(rosenbrock_function, rosenbrock_gradient, x0)

print(f"Starting point: {x0}")
print(f"Optimized point: {x_opt}")
print(f"True minimum: [1.0, 1.0]")
print(f"Final loss: {rosenbrock_function(x_opt):.8f}")
print(f"Number of iterations: {len(optimizer.history['params'])}")

In [ ]:
# Visualize 2D optimization
x_range = np.linspace(-2, 2, 200)
y_range = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rosenbrock_function(np.array([X[i, j], Y[i, j]]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Contour plot
contour = ax1.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
ax1.clabel(contour, inline=True, fontsize=8)

params = np.array(optimizer.history['params'])
ax1.plot(params[:, 0], params[:, 1], 'r.-', linewidth=2, markersize=4, label='Optimization Path')
ax1.plot(params[0, 0], params[0, 1], 'go', markersize=12, label='Start')
ax1.plot(params[-1, 0], params[-1, 1], 'r*', markersize=15, label='End')
ax1.plot(1, 1, 'b*', markersize=15, label='Global Minimum')

ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Optimization Trajectory on Rosenbrock Function', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Loss convergence
ax2.plot(optimizer.history['loss'], 'b-', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Loss Convergence', fontsize=13)
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Effect of Learning Rate

The learning rate $\eta$ is crucial:
- **Too small**: slow convergence
- **Too large**: oscillation or divergence
- **Just right**: fast and stable convergence

In [ ]:
learning_rates = [0.0001, 0.001, 0.01]
x0 = np.array([-1.0, -1.0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, lr in enumerate(learning_rates):
    optimizer = BatchGradientDescent(learning_rate=lr, max_iterations=1000, tolerance=1e-8)
    x_opt = optimizer.optimize(rosenbrock_function, rosenbrock_gradient, x0)
    
    axes[i].plot(optimizer.history['loss'], linewidth=2)
    axes[i].set_xlabel('Iteration', fontsize=11)
    axes[i].set_ylabel('Loss', fontsize=11)
    axes[i].set_title(f'Learning Rate: {lr}', fontsize=12)
    axes[i].set_yscale('log')
    axes[i].grid(True, alpha=0.3)
    
    final_loss = rosenbrock_function(x_opt)
    axes[i].text(0.5, 0.95, f'Final Loss: {final_loss:.4f}',
                transform=axes[i].transAxes, ha='center', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Gradient descent** iteratively moves in the direction of steepest descent: $x \leftarrow x - \eta \nabla f(x)$

2. **Learning rate** is critical:
   - Controls step size
   - Affects convergence speed and stability
   - Often needs tuning

3. **Convergence** depends on:
   - Function landscape (convex vs non-convex)
   - Starting point
   - Learning rate
   - Number of iterations

4. For **deep learning** (next notebook), we use:
   - Mini-batch gradient descent
   - Adaptive learning rates (Adam)
   - Gradient clipping
   - Momentum